In [1]:
import torch
from torch import nn
import torch_geometric
from torch_geometric.nn import SAGEConv, GraphConv, GraphSAGE
import torch
import networkx as nx 
from torch.utils.data import Dataset, DataLoader
import pickle
import pdb 
import numpy as np
from torch_geometric.data import Data
import networkx as nx
import numpy as np
import pandas as pd 
import pickle
import pdb
from torch_geometric.loader import DataLoader
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.utils import to_dense_adj, subgraph, k_hop_subgraph
import matplotlib.pyplot as plt
import sys
import os
sys.path.append(os.path.join(os.path.dirname(sys.path[0]),'tools'))
from sklearn.neighbors import kneighbors_graph
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.data import Data
from newGraphDat import GraphDataset
# from graphDat import GraphDataset
# from ClassGraphDat import GraphDataset

torch.manual_seed(42)

In [2]:

# class GraphEncoderWithResidual(nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels):
#         super(GraphEncoderWithResidual, self).__init__()
#         self.conv1 = SAGEConv(in_channels, hidden_channels*2)
#         self.conv2 = SAGEConv(hidden_channels*2, hidden_channels)
#         # self.conv3 = SAGEConv(hidden_channels, out_channels)
#         self.lin = nn.Linear(hidden_channels, out_channels)
#         # Linear transformation to match dimensions for residual connection
#         self.shortcut = nn.Linear(in_channels, out_channels)
#         # self.sm = nn.Softmax(dim=1)

#     def forward(self, x, edge_index):
#         identity = x
#         x = F.relu(self.conv1(x, edge_index))
#         x = F.relu(self.conv2(x, edge_index))
#         # x = self.conv3(x, edge_index)
#         x = self.lin(x)
#         # Applying shortcut and adding it to the output of conv3
#         identity = self.shortcut(identity)
#         x += identity  # Element-wise addition
#         # print('before: ', x[0])
#         # print(np.shape(x))
#         # x = self.sm(x)
#         # print('after: ', x[0])
#         return x
    
# class GraphEncoderWithResidual(nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels):
#         super(GraphEncoderWithResidual, self).__init__()
#         self.conv1 = SAGEConv(in_channels, hidden_channels*4)
#         self.conv2 = SAGEConv(hidden_channels*4, hidden_channels*2)
#         self.conv3 = SAGEConv(hidden_channels*2, hidden_channels)
#         self.lin1 = nn.Linear(hidden_channels, int(hidden_channels/2))
#         self.lin2 = nn.Linear(int(hidden_channels/2), out_channels)
#         # Linear transformation to match dimensions for residual connection
#         self.shortcut = nn.Linear(in_channels, out_channels)
#         # self.sm = nn.Softmax(dim=1)

#     def forward(self, x, edge_index):
#         identity = x
#         x = F.relu(self.conv1(x, edge_index))
#         x = F.relu(self.conv2(x, edge_index))
#         x = self.conv3(x, edge_index)
#         x = F.relu(self.lin1(x))
#         x = F.dropout(x, p=0.2)
#         x = self.lin2(x)
#         # Applying shortcut and adding it to the output of conv3
#         identity = self.shortcut(identity)
#         x += identity  # Element-wise addition
#         # x = self.sm(x)
#         return x
    

class GraphEncoderWithResidual(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphEncoderWithResidual, self).__init__()
        # self.conv1 = SAGEConv(in_channels, hidden_channels*4, flow="source_to_target", root_weight=False)
        # self.conv2 = SAGEConv(hidden_channels*4, hidden_channels*2, flow="source_to_target", root_weight=False)
        # self.conv3 = SAGEConv(hidden_channels*2, hidden_channels, flow="source_to_target", root_weight=False)
        self.conv1 = SAGEConv(in_channels, hidden_channels*4, flow="target_to_source", root_weight=False)
        self.conv2 = SAGEConv(hidden_channels*4, hidden_channels*2, flow="target_to_source", root_weight=False)
        self.conv3 = SAGEConv(hidden_channels*2, hidden_channels, flow="target_to_source", root_weight=False)
        self.lin1 = nn.Linear(hidden_channels, int(hidden_channels/2))
        self.lin2 = nn.Linear(int(hidden_channels/2), out_channels)
        # Linear transformation to match dimensions for residual connection
        self.shortcut = nn.Linear(in_channels, out_channels)
        # self.sm = nn.Softmax(dim=1)

    def forward(self, x, edge_index):
        identity = x
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = self.conv3(x, edge_index)
        x = F.dropout(F.relu(self.lin1(x)), p=0.5)
        x = self.lin2(x)
        # Applying shortcut and adding it to the output of conv3
        identity = self.shortcut(identity)
        x += identity  # Element-wise addition
        # x = self.sm(x)
        return x

In [3]:
# # Assuming a loss function appropriate for node feature reconstruction, e.g., MSE for continuous features
# def loss_function(reconstructed_x, original_x):
#     # print(/)
#     return F.cross_entropy(reconstructed_x, original_x)

def cosine_loss(adj, emb):
    norms = torch.norm(emb, p=2, dim=1, keepdim=True)
    normalized_embeddings = emb / norms.clamp(min=1e-4)
    cosine_similarity_matrix = torch.mm(normalized_embeddings, normalized_embeddings.t())
    target_adjacency = torch.zeros_like(cosine_similarity_matrix)
    target_adjacency[adj[0], adj[1]] = 1
    # Directly using logits; no need to apply sigmoid.
    return F.binary_cross_entropy_with_logits(cosine_similarity_matrix, target_adjacency)

In [4]:
def train(model, data_loader, optimizer, device):
    model.train()
    total_loss = 0
    embeddings = []
    # origs = []
    # lossfunc = nn.CrossEntropyLoss()
    # print(data_loader)
    for feat, edge, _, glob in data_loader:
        optimizer.zero_grad()
        # pdb.set_trace()
        # print(np.shape(feat), np.shape(edges))
        # print('in main: ', np.shape(feat), np.shape(edges))
        features = feat.squeeze_(0).to(device)
        globalInfo = glob.squeeze_(0).to(device)
        edges = edge.squeeze_(0).to(device)
        embed = model(torch.cat([globalInfo, features], dim=1), edges)
        # origs.append(data[0][0].detach())
        # print(np.shape(embed), np.shape(data[2][0]))
        # print()
        # print(data[2][0][0], embed[0].detach())
        # loss = lossfunc(embed, data[2][0].to(device))
        # print(np.shape(embed), np.shape(data[2][0]), np.shape(data[0][0]), np.shape(data[1][0]))
        # loss = loss_function(embed, data[2][0].to(device)) #cosine_loss(data[1][0].to(device), embed)
        loss = cosine_loss(edges, embed)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        embeddings.append(embed.detach())
        # print(total_loss)
    return total_loss / len(data_loader), embeddings#, origs

In [5]:
# data = torch.load('finished_graph.pth')
# print(data)


In [6]:
# np.shape(data['feat'][1])

In [7]:
# folder_graph = '../CDC/allenvsmore/'
device = "cpu"
# fname = 'small_graphs.pth'
# files = os.listdir(folder_graph)
# files = [file for file in files if file.endswith('.pickle')]
hC = 100
inC = 44


# data = torch.load('finished_graph.pth')

# Create a graph data object
# graph = Data(x=node_features, edge_index=adjacency_matrix.coalesce().indices(), edge_attr=edge_features)

dataset = GraphDataset('all_time_classes_quals.pickle')
print('data loaded')
print(dataset)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=8)
actuals = []
final_loss = []
embeddings = []
outchannels = 3

# torch.save(decoder_state_dict, './CDC/decoder_state_dict'+str(outchannels)+'.pth')
        # print(np.shape(embed), np.shape(data[2][0]), np.shape(data[0][0]), np.shape(data[1][0]))


data loaded


In [8]:
# print(len(dataset))
# for feat, edges, _, _  in dataloader:
#     print(np.shape(feat), np.shape(edges))
    # break

In [9]:
model = GraphEncoderWithResidual(in_channels=inC, hidden_channels=hC, out_channels=outchannels).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.1)
# scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3, verbose=True)
# losses = []
embeds = []
torch.autograd.set_detect_anomaly(True)
# acts = []
for epoch in range(1, 5):  # Number of epochs
    # loss, embed, orig = train(model, dataloader, optimizer, device)
    loss, _ = train(model, dataloader, optimizer, device)
    # embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
    # scheduler.step(loss)
# epoch_loss = loss / len(dataloader)
# embeddings.append(embeds)
# final_loss.append(losses)
# actuals.append(acts)
model.eval()
# Assuming encoder and decoder are your model's components
encoder_state_dict = model.state_dict()
# decoder_state_dict = model.decoder.state_dict()
# Save the state dictionaries
torch.save(encoder_state_dict, './Finished_Large_LinNew_encoder_state_dict'+str(outchannels)+'.pth')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (141x104 and 44x400)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.01)
# scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3, verbose=True)
for epoch in range(1, 5):  # Number of epochs
    # loss, embed, orig = train(model, dataloader, optimizer, device)
    loss, _ = train(model, dataloader, optimizer, device)
    # embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
    # scheduler.step(loss)
model.eval()
encoder_state_dict = model.state_dict()
# decoder_state_dict = model.decoder.state_dict()
# Save the state dictionaries
torch.save(encoder_state_dict, './Finished_Large_LinNew_encoder_state_dict'+str(outchannels)+'.pth')

Epoch 1, Loss: 0.7854
Epoch 2, Loss: 0.7610
Epoch 3, Loss: 0.7605
Epoch 4, Loss: 0.7458


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
# scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3, verbose=True)
for epoch in range(1, 5):  # Number of epochs
    # loss, embed, orig = train(model, dataloader, optimizer, device)
    loss, embed = train(model, dataloader, optimizer, device)
    embeds.append(embed)
    # acts.append(orig)
    print(f'Epoch {epoch}, Loss: {loss:.4f}')
    # losses.append(loss)
    # scheduler.step(loss)
model.eval()
encoder_state_dict = model.state_dict()
# decoder_state_dict = model.decoder.state_dict()
# Save the state dictionaries
torch.save(encoder_state_dict, './Finished_Large_LinNew_encoder_state_dict'+str(outchannels)+'.pth')

Epoch 1, Loss: 0.7518
Epoch 2, Loss: 0.7444
Epoch 3, Loss: 0.7514
Epoch 4, Loss: 0.7479


In [ ]:
# for i in embeddings:
#     df = pd.DataFrame(emb, columns=['x', 'y', 'z'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_emb.csv')

#     df = pd.DataFrame(arrpos, columns=['pose1', 'pose2', 'pose3', 'pose4'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_poses.csv')

#     df = pd.DataFrame(arrqual, columns=['qual1', 'qual2', 'qual3', 'qual4'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_quals.csv')


#     df = pd.DataFrame(arr_global_info, columns=['1', '2', '3', '4'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_global_info.csv')

#     df = pd.DataFrame(arrcol, columns=['colors'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_colors.csv')

#     df = pd.DataFrame(arr_num_agents, columns=['num_agents'])
#     df.to_csv('./RS/LinNew_ALLENVS_AGENTS_num_agents.csv')
# optimizer = optim.Adam(model.parameters(), lr=0.0001)
# # scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.1, patience=3, verbose=True)
# for epoch in range(1, 5):  # Number of epochs
#     loss, embed, orig = train(model, dataloader, optimizer, device)
#     # embeds.append(embed)
#     # acts.append(orig)
#     print(f'Epoch {epoch}, Loss: {loss:.4f}')
#     # losses.append(loss)
#     # scheduler.step(loss)
# encoder_state_dict = model.state_dict()
# # decoder_state_dict = model.decoder.state_dict()
# # Save the state dictionaries
# torch.save(encoder_state_dict, './LinNew_encoder_state_dict'+str(outchannels)+'.pth')

In [ ]:


# sf = nn.Softmax(dim=1)
# x = torch.tensor([[-0.5207, -0.6732,  0.1790, -0.5166], [-0.3699, -0.7055, -0.0155, -0.2880]])
# print(np.shape(x))
# print(sf(x))

In [ ]:
# import torch
# import torch.nn.functional as F
# # Example of target with class indices
# input = torch.randn(3, 5, requires_grad=True)
# print(input)
# target = torch.randint(5, (3,), dtype=torch.int64)
# print(target)
# loss = F.cross_entropy(input, target)
# print(loss)
# loss.backward()
# print(loss)

# # Example of target with class probabilities
# input = torch.randn(3, 5, requires_grad=True)
# target = torch.randn(3, 5).softmax(dim=1)
# print(input)
# print(target)
# loss = F.cross_entropy(input, target)
# print(loss)
# loss.backward()
# print(loss)

# q0 >= q1 >= q2 >= q3
# [q0 site recruiters, q1 site recruiters, q2 site recruiters, q3 site recruiters, agent state, site x, site y, site quality ...]